In [ ]:
import os

import pandas as pd
import seaborn as sns

In [ ]:
rdfs = []
for dir in os.listdir("modal-results/"):
    _, _, gpu, run = dir.split("-", maxsplit=3)
    try:
        rdf = pd.read_csv(f"modal-results/{dir}/fused-mm-sample-batch-scaling.csv")
    except FileNotFoundError:
        continue
    rdf_long = rdf.melt(id_vars=["n_hidden_states"], var_name="Method", value_name="Time[ms]")
    rdf_long = rdf_long.assign(gpu=gpu, run=run)
    rdfs.append(rdf_long)
rdf_long = pd.concat(rdfs).astype({"n_hidden_states": int})
rdf_long["branch"] = rdf_long["run"].str[:-1]
rdf_long

In [ ]:
sns.set_context("talk")
axs = sns.catplot(
    # rdf_long.query("n_hidden_states < 256 and Method == 'flashinfer:sampling_from_logits'"),
    rdf_long.query("n_hidden_states < 256 and Method == 'FMMS (Triton)'"),
    x="n_hidden_states",
    y="Time[ms]",
    hue="branch",
    col="gpu",
    kind="strip",
    errorbar=lambda xs: (min(xs), max(xs)),
    sharey=False,
)
for ax in axs.axes.flat:
    ax.grid(alpha=0.5, axis="y")

# Best configs

In [ ]:
!grep -r "best config" ./modal-results/*/logs.txt